In [0]:
from pyspark.sql.functions import col, when, round, current_timestamp, lit
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import udf
import mlflow
import pandas as pd

# Load gold table
df_gold = spark.table("gold_db.gold_bank_features").fillna(0)

print("Gold table loaded:", df_gold.count(), "banks")
print("Failed banks:", df_gold.filter("failed = 1").count())
print("Active banks:", df_gold.filter("failed = 0").count())

Gold table loaded: 108 banks
Failed banks: 8
Active banks: 100


In [0]:
# Define feature columns
feature_cols = [
    "avg_asset", "avg_dep", "avg_equity", "avg_net_income",
    "avg_capital_ratio", "avg_loan_to_deposit", "avg_profit_margin",
    "avg_leverage_ratio", "avg_deposit_to_asset",
    "pagerank", "degree_centrality", "betweenness_centrality",
    "clustering_coefficient", "asset_volatility", "income_volatility"
]

# Assemble features into single vector
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assembler.transform(df_gold).select("CERT", "features", "failed", "risk_tier")

# Handle class imbalance
df_active = df_ml.filter("failed = 0")
df_failed = df_ml.filter("failed = 1")
df_failed_oversampled = df_failed.sample(withReplacement=True, fraction=5.0, seed=42)
df_balanced = df_active.union(df_failed_oversampled)

print("Balanced dataset:")
print("Active banks:", df_balanced.filter("failed = 0").count())
print("Failed banks:", df_balanced.filter("failed = 1").count())

Balanced dataset:
Active banks: 100
Failed banks: 29


In [0]:
# Split data
train_df, test_df = df_balanced.randomSplit([0.8, 0.2], seed=42)

with mlflow.start_run(run_name="random_forest_v1"):

    # Train
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="failed",
        numTrees=100,
        maxDepth=5,
        seed=42
    )
    model = rf.fit(train_df)
    predictions = model.transform(test_df)

    # Evaluate
    evaluator_auc = BinaryClassificationEvaluator(
        labelCol="failed",
        metricName="areaUnderROC"
    )
    evaluator = MulticlassClassificationEvaluator(
        labelCol="failed",
        predictionCol="prediction"
    )

    auc       = evaluator_auc.evaluate(predictions)
    accuracy  = evaluator.setMetricName("accuracy").evaluate(predictions)
    precision = evaluator.setMetricName("weightedPrecision").evaluate(predictions)
    recall    = evaluator.setMetricName("weightedRecall").evaluate(predictions)
    f1        = evaluator.setMetricName("f1").evaluate(predictions)

    # Log to MLflow
    mlflow.log_param("num_trees", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_param("train_size", train_df.count())
    mlflow.log_param("test_size", test_df.count())
    mlflow.log_metric("auc_roc", float(auc))
    mlflow.log_metric("accuracy", float(accuracy))
    mlflow.log_metric("precision", float(precision))
    mlflow.log_metric("recall", float(recall))
    mlflow.log_metric("f1_score", float(f1))

# Print outside mlflow context
import builtins
r = builtins.round

print("Model Evaluation Results:")
print("="*40)
print("Accuracy  :", r(accuracy, 4))
print("Precision :", r(precision, 4))
print("Recall    :", r(recall, 4))
print("F1 Score  :", r(f1, 4))
print("AUC-ROC   :", r(auc, 4))
print("="*40)

Model Evaluation Results:
Accuracy  : 0.9583
Precision : 0.9635
Recall    : 0.9583
F1 Score  : 0.9591
AUC-ROC   : 1.0


In [0]:
import pandas as pd

# Confusion matrix
print("Confusion Matrix:")
predictions.groupBy("failed", "prediction").count().show()

# Feature importance
feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.featureImportances.toArray()
}).sort_values("importance", ascending=False)

print("Top 10 Important Features:")
print(feature_importance.head(10).to_string(index=False))

Confusion Matrix:
+------+----------+-----+
|failed|prediction|count|
+------+----------+-----+
|     0|       0.0|   16|
|     0|       1.0|    1|
|     1|       1.0|    7|
+------+----------+-----+

Top 10 Important Features:
               feature  importance
     avg_profit_margin    0.172998
     income_volatility    0.153993
        avg_net_income    0.149838
clustering_coefficient    0.107370
             avg_asset    0.084320
      asset_volatility    0.055720
            avg_equity    0.051816
  avg_deposit_to_asset    0.041373
               avg_dep    0.039146
betweenness_centrality    0.035996


In [0]:
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import udf, current_timestamp, lit
import builtins

get_failure_prob = udf(lambda v: float(v[1]), DoubleType())

df_predictions = predictions \
    .withColumn("failure_probability", get_failure_prob("probability")) \
    .select(
        "CERT",
        "risk_tier",
        "failed",
        "prediction",
        "failure_probability"
    ) \
    .withColumn("prediction_timestamp", current_timestamp()) \
    .withColumn("model_version", lit("random_forest_v1"))

df_predictions.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_db.gold_predictions")

print("gold_predictions saved:", spark.table("gold_db.gold_predictions").count(), "rows")
display(spark.table("gold_db.gold_predictions").orderBy("failure_probability", ascending=False).limit(10))

gold_predictions saved: 24 rows


CERT,risk_tier,failed,prediction,failure_probability,prediction_timestamp,model_version
10086,HIGH,1,1.0,0.9655541958041958,2026-03-13T16:51:44.435Z,random_forest_v1
10086,HIGH,1,1.0,0.9655541958041958,2026-03-13T16:51:44.435Z,random_forest_v1
10054,MEDIUM,1,1.0,0.942395946159104,2026-03-13T16:51:44.435Z,random_forest_v1
10054,MEDIUM,1,1.0,0.942395946159104,2026-03-13T16:51:44.435Z,random_forest_v1
10094,HIGH,1,1.0,0.9108766233766233,2026-03-13T16:51:44.435Z,random_forest_v1
10155,MEDIUM,1,1.0,0.8213606302593025,2026-03-13T16:51:44.435Z,random_forest_v1
1006,MEDIUM,1,1.0,0.7461282505470568,2026-03-13T16:51:44.435Z,random_forest_v1
10044,LOW,0,1.0,0.5200712007290955,2026-03-13T16:51:44.435Z,random_forest_v1
1003,LOW,0,0.0,0.4407854864433812,2026-03-13T16:51:44.435Z,random_forest_v1
1012,HIGH,0,0.0,0.14515708380575612,2026-03-13T16:51:44.435Z,random_forest_v1
